# Zepto Data Engineering Pipeline: Catalog Scraper & Relational Store

**Module 1 — Data Pipeline (/data_pipeline)**
*Author:* AI/ML Engineering Team, Zepto

---

### Objectives:
1. **Scrape** live catalog data from `books.toscrape.com` across $\ge 3$ product categories ($\ge 60$ books) capturing title, price (GBP), star rating, availability, and category.
2. **Clean & Enrich**: Strip currency symbols, parse ratings to integers (1–5), cast availability to boolean, and convert GBP to INR using the **fixed baseline conversion rate (1 GBP = 105.50 INR)**.
3. **Database Design & Ingestion**: Build a normalized 2-table SQLite database (`categories` and `books`) with Primary Key / Foreign Key constraints.
4. **SQL Analytics**: Execute $\ge 5$ SQL queries demonstrating `SELECT/WHERE`, `ORDER BY`, `LIMIT`, `DISTINCT`, `BETWEEN`, and a relational `JOIN`.
5. **Pandas Validation**: Query database via `pd.read_sql` and reproduce the relational join in-memory via `pd.merge()`, verifying mathematical and structural equivalence.


In [1]:
import sys
import os
import re
import sqlite3
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import urllib.parse

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("Requests version:", requests.__version__)


Python version: 3.10.10 (tags/v3.10.10:aad5f6a, Feb  7 2023, 17:20:36) [MSC v.1929 64 bit (AMD64)]
Pandas version: 2.3.3
Requests version: 2.34.2


## Step 1: Web Scraping (`books.toscrape.com`)

We fetch product data across 4 diverse categories: `Travel`, `Mystery`, `Historical Fiction`, and `Sequential Art`.


In [2]:
BASE_URL = "http://books.toscrape.com/"

def get_category_links(base_url=BASE_URL):
    resp = requests.get(base_url, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    categories = {}
    for link in soup.select("div.side_categories ul li ul li a"):
        categories[link.text.strip()] = urllib.parse.urljoin(base_url, link.get("href"))
    return categories

def scrape_category_books(category_name, category_url):
    books = []
    current_url = category_url
    while current_url:
        resp = requests.get(current_url, timeout=15)
        if resp.status_code != 200:
            break
        soup = BeautifulSoup(resp.text, "html.parser")
        for art in soup.select("article.product_pod"):
            title_tag = art.select_one("h3 a")
            title = title_tag.get("title") if title_tag and title_tag.get("title") else (title_tag.text.strip() if title_tag else "Unknown")
            
            price_tag = art.select_one("p.price_color")
            price_raw = price_tag.text.strip() if price_tag else "0.00"
            
            rating_tag = art.select_one("p.star-rating")
            rating_classes = rating_tag.get("class", []) if rating_tag else []
            rating_text = "Zero"
            for cls in rating_classes:
                if cls.lower() != "star-rating":
                    rating_text = cls
                    break
            
            avail_tag = art.select_one("p.instock.availability")
            avail_text = avail_tag.text.strip() if avail_tag else "In stock"
            
            books.append({
                "title": title,
                "price": price_raw,
                "star_rating": rating_text,
                "availability": avail_text,
                "category": category_name
            })
        next_button = soup.select_one("li.next a")
        current_url = urllib.parse.urljoin(current_url, next_button.get("href")) if next_button else None
    return books

all_cats = get_category_links()
target_cats = ["Travel", "Mystery", "Historical Fiction", "Sequential Art"]

raw_books = []
for cat in target_cats:
    if cat in all_cats:
        c_books = scrape_category_books(cat, all_cats[cat])
        print(f"Scraped {len(c_books)} books from category: '{cat}'")
        raw_books.extend(c_books)

print(f"\nTotal books scraped: {len(raw_books)} across {len(target_cats)} categories.")


Scraped 11 books from category: 'Travel'


Scraped 32 books from category: 'Mystery'


Scraped 26 books from category: 'Historical Fiction'


Scraped 75 books from category: 'Sequential Art'

Total books scraped: 144 across 4 categories.


## Step 2: Data Cleaning, Typing & Currency Conversion

### Fixed-Rate Currency Conversion
- Rate: **1 GBP = 105.50 INR** (project baseline constant)
- Imputation Decision: Any unparsable numeric prices are imputed using the median price of the catalog.


In [3]:
EXCHANGE_RATE_GBP_TO_INR = 105.50
RATING_MAP = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5}

df = pd.DataFrame(raw_books)

# 1. Price GBP
def parse_price(val):
    if pd.isna(val): return np.nan
    m = re.search(r"(\d+\.?\d*)", str(val))
    return float(m.group(1)) if m else np.nan

df["price_gbp"] = df["price"].apply(parse_price)
if df["price_gbp"].isna().any():
    df["price_gbp"] = df["price_gbp"].fillna(df["price_gbp"].median())
df["price_gbp"] = df["price_gbp"].round(2)

# 2. Price INR
df["price_inr"] = (df["price_gbp"] * EXCHANGE_RATE_GBP_TO_INR).round(2)

# 3. Rating
df["rating"] = df["star_rating"].apply(lambda v: RATING_MAP.get(str(v).strip().lower(), 3)).astype(int)

# 4. In Stock
df["in_stock"] = df["availability"].apply(lambda v: "in stock" in str(v).lower()).astype(bool)

cleaned_df = df[["title", "category", "price_gbp", "price_inr", "rating", "in_stock"]].copy()
print("Cleaned Dataset Info:")
print(cleaned_df.info())
cleaned_df.head(10)


Cleaned Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      144 non-null    object 
 1   category   144 non-null    object 
 2   price_gbp  144 non-null    float64
 3   price_inr  144 non-null    float64
 4   rating     144 non-null    int64  
 5   in_stock   144 non-null    bool   
dtypes: bool(1), float64(2), int64(1), object(2)
memory usage: 5.9+ KB
None


,title,category,price_gbp,price_inr,rating,in_stock
0,It's Only the Himalayas,Travel,45.17,4765.44,2,True
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Travel,49.43,5214.86,4,True
2,See America: A Celebration of Our National Par...,Travel,48.87,5155.78,3,True
3,Vagabonding: An Uncommon Guide to the Art of L...,Travel,36.94,3897.17,2,True
4,Under the Tuscan Sun,Travel,37.33,3938.31,3,True
5,A Summer In Europe,Travel,44.34,4677.87,2,True
6,The Great Railway Bazaar,Travel,30.54,3221.97,1,True
7,A Year in Provence (Provence #1),Travel,56.88,6000.84,4,True
8,The Road to Little Dribbling: Adventures of an...,Travel,23.21,2448.66,1,True
9,Neither Here nor There: Travels in Europe,Travel,38.95,4109.23,3,True


## Step 3: Normalized SQLite Database Storage

We construct a normalized schema with two tables:
1. `categories(category_id PK, category_name UNIQUE)`
2. `books(book_id PK, title, price_gbp, price_inr, rating, in_stock, category_id FK -> categories)`


In [4]:
db_path = "books.db"
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)
cur = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON;")

cur.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id) REFERENCES categories (category_id) ON DELETE CASCADE
);
""")
conn.commit()

# Load categories
unique_categories = sorted(cleaned_df["category"].unique())
for cat in unique_categories:
    cur.execute("INSERT OR IGNORE INTO categories (category_name) VALUES (?)", (cat,))
conn.commit()

cat_df = pd.read_sql("SELECT category_id, category_name FROM categories", conn)
cat_map = dict(zip(cat_df["category_name"], cat_df["category_id"]))

# Load books
books_tuples = [
    (row["title"], float(row["price_gbp"]), float(row["price_inr"]), int(row["rating"]), 1 if row["in_stock"] else 0, cat_map[row["category"]])
    for _, row in cleaned_df.iterrows()
]

cur.executemany("""
INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
VALUES (?, ?, ?, ?, ?, ?)
""", books_tuples)
conn.commit()

print(f"Database populated with {len(unique_categories)} categories and {len(books_tuples)} books.")


Database populated with 4 categories and 144 books.


## Step 4: SQL Queries & Analysis

Executing 5 SQL queries covering:
- `SELECT` / `WHERE`
- `ORDER BY`
- `LIMIT`
- `DISTINCT`
- `BETWEEN`
- `JOIN` (Relational Inner Join)


In [5]:
# Query 1: SELECT, WHERE, ORDER BY
q1 = """
SELECT title, price_gbp, price_inr, rating
FROM books
WHERE rating >= 4 AND in_stock = 1
ORDER BY rating DESC, price_gbp ASC;
"""
df_q1 = pd.read_sql_query(q1, conn)
print("=== Query 1: Books with Rating >= 4 (In Stock) ===")
df_q1.head(5)


=== Query 1: Books with Rating >= 4 (In Stock) ===


,title,price_gbp,price_inr,rating
0,"Fruits Basket, Vol. 2 (Fruits Basket #2)",11.64,1228.02,5
1,Superman Vol. 1: Before Truth (Superman by Gen...,11.89,1254.40,5
2,The Girl You Lost,12.29,1296.59,5
3,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,1435.86,5
4,Roller Girl,14.10,1487.55,5


In [6]:
# Query 2: ORDER BY, LIMIT
q2 = """
SELECT title, price_gbp, price_inr, rating
FROM books
ORDER BY price_inr DESC
LIMIT 5;
"""
df_q2 = pd.read_sql_query(q2, conn)
print("=== Query 2: Top 5 Most Expensive Books ===")
df_q2


=== Query 2: Top 5 Most Expensive Books ===


,title,price_gbp,price_inr,rating
0,Boar Island (Anna Pigeon #19),59.48,6275.14,3
1,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,57.70,6087.35,4
2,El Deafo,57.62,6078.91,5
3,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",57.06,6019.83,4
4,A Year in Provence (Provence #1),56.88,6000.84,4


In [7]:
# Query 3: DISTINCT
q3 = """
SELECT DISTINCT rating
FROM books
ORDER BY rating ASC;
"""
df_q3 = pd.read_sql_query(q3, conn)
print("=== Query 3: Distinct Star Ratings in Catalog ===")
df_q3


=== Query 3: Distinct Star Ratings in Catalog ===


,rating
0,1
1,2
2,3
3,4
4,5


In [8]:
# Query 4: BETWEEN
q4 = """
SELECT title, price_inr, rating
FROM books
WHERE price_inr BETWEEN 2000.0 AND 4000.0
ORDER BY price_inr ASC;
"""
df_q4 = pd.read_sql_query(q4, conn)
print("=== Query 4: Books Priced Between INR 2,000 and INR 4,000 ===")
df_q4.head(5)


=== Query 4: Books Priced Between INR 2,000 and INR 4,000 ===


,title,price_inr,rating
0,"Pop Gun War, Volume 1: Gift",2001.33,1
1,The Cuckoo's Calling (Cormoran Strike #1),2026.66,1
2,This One Summer,2056.19,4
3,"Fruits Basket, Vol. 7 (Fruits Basket #7)",2064.64,1
4,"In a Dark, Dark Wood",2070.96,1


In [9]:
# Query 5: Relational JOIN with Grouping and Aggregation
q5 = """
SELECT 
    c.category_name,
    COUNT(b.book_id) AS total_books,
    ROUND(AVG(b.price_gbp), 2) AS avg_price_gbp,
    ROUND(AVG(b.price_inr), 2) AS avg_price_inr,
    MAX(b.rating) AS max_rating
FROM categories c
INNER JOIN books b ON c.category_id = b.category_id
GROUP BY c.category_id, c.category_name
ORDER BY total_books DESC, avg_price_inr DESC;
"""
df_q5 = pd.read_sql_query(q5, conn)
print("=== Query 5: Category Summary via SQL JOIN ===")
df_q5


=== Query 5: Category Summary via SQL JOIN ===


,category_name,total_books,avg_price_gbp,avg_price_inr,max_rating
0,Sequential Art,75,34.57,3647.37,5
1,Mystery,32,31.72,3346.36,5
2,Historical Fiction,26,33.64,3549.47,5
3,Travel,11,39.79,4198.32,5


## Step 5: Pandas Merge Equivalence Verification

We read the raw tables into memory and replicate the join + aggregation logic strictly using pandas methods (`pd.merge`, `groupby`, `agg`).


In [10]:
df_books = pd.read_sql("SELECT * FROM books", conn)
df_categories = pd.read_sql("SELECT * FROM categories", conn)

merged = pd.merge(df_categories, df_books, on="category_id", how="inner")
df_pandas = merged.groupby(["category_id", "category_name"]).agg(
    total_books=("book_id", "count"),
    avg_price_gbp=("price_gbp", lambda x: round(x.mean(), 2)),
    avg_price_inr=("price_inr", lambda x: round(x.mean(), 2)),
    max_rating=("rating", "max")
).reset_index().drop(columns=["category_id"])

df_pandas = df_pandas.sort_values(by=["total_books", "avg_price_inr"], ascending=[False, False]).reset_index(drop=True)

print("--- SQL JOIN Output ---")
display(df_q5)

print("--- Pandas Merge Output ---")
display(df_pandas)

# Assert frames are identical
pd.testing.assert_frame_equal(df_q5, df_pandas, check_dtype=False)
print("\n[SUCCESS] Exact equivalence confirmed between SQL JOIN and pd.merge()!")


--- SQL JOIN Output ---


,category_name,total_books,avg_price_gbp,avg_price_inr,max_rating
0,Sequential Art,75,34.57,3647.37,5
1,Mystery,32,31.72,3346.36,5
2,Historical Fiction,26,33.64,3549.47,5
3,Travel,11,39.79,4198.32,5


--- Pandas Merge Output ---


,category_name,total_books,avg_price_gbp,avg_price_inr,max_rating
0,Sequential Art,75,34.57,3647.37,5
1,Mystery,32,31.72,3346.36,5
2,Historical Fiction,26,33.64,3549.47,5
3,Travel,11,39.79,4198.32,5



[SUCCESS] Exact equivalence confirmed between SQL JOIN and pd.merge()!


In [11]:
conn.close()
print("Database connection closed cleanly.")


Database connection closed cleanly.
